In [15]:
import pandas as pd
import os
import json

# 1. Load label mapping từ file
csv_file = 'data/hrv_features_label.csv'
df_labels = pd.read_csv(csv_file)

# Strip column names to handle manual entry spaces
df_labels.columns = df_labels.columns.str.strip()

print("=" * 80)
print("📊 Loading Label Mapping from CSV")
print("=" * 80)
print(f"\n✅ Loaded: {csv_file}")
print(f"   Rows: {len(df_labels)}")
print(f"   Columns: {list(df_labels.columns)}\n")

# Display first few rows
print("📋 First few rows:")
print(df_labels.head(10))
print()

📊 Loading Label Mapping from CSV

✅ Loaded: data/hrv_features_label.csv
   Rows: 46
   Columns: ['File', 'Mean_RR(ms)', 'HR(bpm)', 'SDNN(ms)', 'RMSSD(ms)', 'pNN50(%)', 'LF Power', 'HF Power', 'LF/HF Ratio', 'Label']

📋 First few rows:
        File  Mean_RR(ms)     HR(bpm)   SDNN(ms)  RMSSD(ms)   pNN50(%)  \
0  hr100.csv   599.715074  100.047510  13.330608   4.932311   0.000000   
1  hr101.csv   593.822674  101.040264  12.918245   4.767402   0.000000   
2  hr102.csv   587.971630  102.045740  12.503373   4.670479   0.000000   
3  hr103.csv   582.200699  103.057245  12.061133   4.550369   0.000000   
4  hr104.csv   576.719810  104.036655  11.805241   4.460409   0.000000   
5  hr105.csv   571.230075  105.036486  11.474376   4.427616   0.000000   
6   hr60.csv   994.949341   60.304578  74.953642  47.969004  32.941176   
7   hr61.csv   978.530649   61.316424  72.032188  45.180965  32.046332   
8   hr62.csv   962.002841   62.369878  67.056307  41.932086  25.475285   
9   hr63.csv   948.256763

In [16]:
# 2. Extract file and label mapping
data_dir = 'data/raw_gen'

# Debug: print columns
print(f"Columns in df_labels: {list(df_labels.columns)}")

# Create mapping: filename → label
file_label_map = {}
for idx, row in df_labels.iterrows():
    # Extract filename from 'File' column
    filename = row['File']  # e.g., "hr60.csv"
    label = row['Label']     # e.g., 0
    
    file_label_map[filename] = label

print(f"✅ Created mapping: {len(file_label_map)} files")
print(f"   Sample: {list(file_label_map.items())[:5]}\n")

Columns in df_labels: ['File', 'Mean_RR(ms)', 'HR(bpm)', 'SDNN(ms)', 'RMSSD(ms)', 'pNN50(%)', 'LF Power', 'HF Power', 'LF/HF Ratio', 'Label']
✅ Created mapping: 46 files
   Sample: [('hr100.csv', 2), ('hr101.csv', 2), ('hr102.csv', 2), ('hr103.csv', 2), ('hr104.csv', 2)]



In [17]:
# 3. Group files by label
label_files = {}
for filename, label in file_label_map.items():
    if label not in label_files:
        label_files[label] = []
    label_files[label].append(filename)

print("=" * 80)
print("📂 Files grouped by Label:")
print("=" * 80)
for label in sorted(label_files.keys()):
    files = sorted(label_files[label])
    print(f"\n🔹 Label {label}: {len(files)} files")
    print(f"   {files[:3]}...{files[-3:]}")  # Show first 3 and last 3



📂 Files grouped by Label:

🔹 Label 0: 23 files
   ['hr60.csv', 'hr61.csv', 'hr62.csv']...['hr80.csv', 'hr81.csv', 'hr82.csv']

🔹 Label 1: 13 files
   ['hr83.csv', 'hr84.csv', 'hr85.csv']...['hr93.csv', 'hr94.csv', 'hr95.csv']

🔹 Label 2: 10 files
   ['hr100.csv', 'hr101.csv', 'hr102.csv']...['hr97.csv', 'hr98.csv', 'hr99.csv']


In [11]:
def get_file_duration(filepath):
    """
    Calculate duration of ECG file from Time column
    
    Returns:
        tuple: (duration_seconds, num_samples, start_time, end_time)
    """
    df = pd.read_csv(filepath)
    num_samples = len(df)
    
    # Check if 'Time' column exists
    if 'Time' in df.columns:
        start_time = df['Time'].min()
        end_time = df['Time'].max()
        duration_seconds = end_time - start_time
    else:
        # Fallback: assume 1 sample = 1 second
        print(f"⚠️  WARNING: 'Time' column not found in {filepath}")
        print(f"   Using fallback: 1 sample = 1 second")
        duration_seconds = num_samples
        start_time = 0
        end_time = num_samples
    
    return duration_seconds, num_samples, start_time, end_time


# Test with one file
filepath = 'data/raw_gen/hr60.csv'
duration, num_samples, start_time, end_time = get_file_duration(filepath)

print(f"📊 File: {filepath}")
print(f"   Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")
print(f"   Samples: {num_samples}")
print(f"   Start time: {start_time:.2f}s")
print(f"   End time: {end_time:.2f}s")

📊 File: data/raw_gen/hr60.csv
   Duration: 256.67 seconds (4.28 minutes)
   Samples: 65709
   Start time: 0.00s
   End time: 256.67s


In [18]:
# 5. Calculate duration for each label
print("\n" + "=" * 80)
print("📈 Duration Analysis by Label:")
print("=" * 80)

all_stats = {}

for label in sorted(label_files.keys()):
    files = sorted(label_files[label])
    print(f"\n🔹 LABEL {label}:")
    print("-" * 80)
    
    durations = {}
    total_samples = 0
    
    for filename in files:
        filepath = os.path.join(data_dir, filename)
        
        try:
            duration, num_samples, start_time, end_time = get_file_duration(filepath)
            durations[filename] = duration
            total_samples += num_samples
            print(f"  {filename:25s}: {duration:8.2f}s ({num_samples:4d} samples) | {start_time:8.2f}s → {end_time:8.2f}s")
        except FileNotFoundError:
            print(f"  {filename:25s}: ⚠️  FILE NOT FOUND!")
        except Exception as e:
            print(f"  {filename:25s}: ⚠️  ERROR - {str(e)}")
    
    total_duration = sum(durations.values())
    
    print(f"\n  {'TOTAL':25s}: {total_duration:8.2f}s ({total_samples} samples) = {total_duration/60:.2f} minutes")
    print(f"  {'Avg per file':25s}: {total_duration/len(files):.2f}s")
    
    # Store stats
    all_stats[label] = {
        'num_files': len(files),
        'total_duration_seconds': total_duration,
        'total_samples': total_samples,
        'total_duration_minutes': total_duration / 60,
        'avg_duration_per_file': total_duration / len(files),
        'files': files
    }


📈 Duration Analysis by Label:

🔹 LABEL 0:
--------------------------------------------------------------------------------
  hr60.csv                 :   256.67s (65709 samples) |     0.00s →   256.67s
  hr61.csv                 :   256.47s (65657 samples) |     0.00s →   256.47s
  hr62.csv                 :   256.28s (65609 samples) |     0.00s →   256.28s
  hr63.csv                 :   256.02s (65541 samples) |     0.00s →   256.02s
  hr64.csv                 :   256.70s (65715 samples) |     0.00s →   256.70s
  hr65.csv                 :   256.52s (65671 samples) |     0.00s →   256.52s
  hr66.csv                 :   256.32s (65618 samples) |     0.00s →   256.32s
  hr67.csv                 :   256.11s (65566 samples) |     0.00s →   256.11s
  hr68.csv                 :   256.73s (65724 samples) |     0.00s →   256.73s
  hr69.csv                 :   256.56s (65680 samples) |     0.00s →   256.56s
  hr70.csv                 :   256.35s (65627 samples) |     0.00s →   256.35s
  hr71.

In [19]:
# 6. Overall summary
print("\n" + "=" * 80)
print("📊 OVERALL SUMMARY:")
print("=" * 80)

total_all_duration = 0
total_all_samples = 0

for label in sorted(all_stats.keys()):
    stats = all_stats[label]
    total_all_duration += stats['total_duration_seconds']
    total_all_samples += stats['total_samples']
    
    label_names = {0: "No Stress", 1: "Mild Stress", 2: "High Stress"}
    label_name = label_names.get(label, "Unknown")
    
    print(f"\nLabel {label} ({label_name:15s}): {stats['num_files']:2d} files → {stats['total_duration_seconds']:7.0f}s ({stats['total_duration_minutes']:6.2f} min) - {stats['total_samples']:6d} samples")

print(f"\n{'TOTAL':40s}: {sum([s['num_files'] for s in all_stats.values()]):2d} files → {total_all_duration:7.0f}s ({total_all_duration/60:6.2f} min) - {total_all_samples:6d} samples")
print("=" * 80)




📊 OVERALL SUMMARY:

Label 0 (No Stress      ): 23 files →    5898s ( 98.29 min) - 1509792 samples

Label 1 (Mild Stress    ): 13 files →    3333s ( 55.56 min) - 853344 samples

Label 2 (High Stress    ): 10 files →    2563s ( 42.72 min) - 656126 samples

TOTAL                                   : 46 files →   11794s (196.56 min) - 3019262 samples


In [20]:
# 7. Save detailed report
report = {
    'summary': {
        'total_files': sum([s['num_files'] for s in all_stats.values()]),
        'total_duration_seconds': total_all_duration,
        'total_duration_minutes': total_all_duration / 60,
        'total_samples': total_all_samples,
        'csv_source': csv_file
    },
    'by_label': all_stats
}

report_file = 'data_duration_report.json'
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✅ Report saved to: {report_file}")




✅ Report saved to: data_duration_report.json


In [21]:
# 8. Create summary DataFrame
summary_data = []
for label in sorted(all_stats.keys()):
    stats = all_stats[label]
    summary_data.append({
        'Label': label,
        'Num_Files': stats['num_files'],
        'Total_Duration_Seconds': int(stats['total_duration_seconds']),
        'Total_Duration_Minutes': round(stats['total_duration_minutes'], 2),
        'Total_Samples': stats['total_samples'],
        'Avg_Duration_Per_File': round(stats['avg_duration_per_file'], 2)
    })

summary_df = pd.DataFrame(summary_data)
summary_csv = 'data_duration_summary.csv'
summary_df.to_csv(summary_csv, index=False)

print(f"✅ Summary saved to: {summary_csv}\n")
print(summary_df.to_string(index=False))

✅ Summary saved to: data_duration_summary.csv

 Label  Num_Files  Total_Duration_Seconds  Total_Duration_Minutes  Total_Samples  Avg_Duration_Per_File
     0         23                    5897                   98.29        1509792                 256.41
     1         13                    3333                   55.56         853344                 256.41
     2         10                    2562                   42.72         656126                 256.30
